‼️ This notebook must be exectuted from Google Collab ‼️

**Creating a fresh deployment folder**

In [ ]:
import os

# Remove the directory if it exists
if os.path.exists("milvus-CR"):
    !rm -rf milvus-CR

# Create the directory
os.makedirs("milvus-CR", exist_ok=True)

# Change to the new directory
os.chdir("milvus-CR")

# Confirm the change
print(f"Current working directory: {os.getcwd()}")

**Creating your environment variables**

‼️ IMPORTANT BEFORE EXECUTING‼️

Enter your PROJECT_ID and e-mail address used by your Google environment

In [4]:
# Replace with your Google Cloud project ID & e-mail
PROJECT_ID = ' YOUR PROJECT ID '  #@param {type:"string"}
EMAIL=' YOUR ASSOCIATED E-MAIL '

# Specify the region (e.g., us-central1)
REGION = 'europe-west1'  #@param {type:"string"}
ZONE= 'europe-west1-b'  #@param {type:"string"}

# Set the Artifact Registry repository name
REPO_NAME = 'milvus-repo'  #@param {type:"string"}

# Set the image name and tag
IMAGE_NAME = 'milvusdb'
IMAGE_TAG = 'milvus:v2.5.0-beta'

# Define the full image URI
IMAGE_URI = f'{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/{IMAGE_NAME}:{IMAGE_TAG}'

SERVICE_NAME='milvus-service'  #@param {type:"string"}
PORT=19530  # Specify the port for the container to listen on

**Authenticate on Google Cloud**

In [6]:
from google.colab import auth
auth.authenticate_user()

**Set your Google config**

In [ ]:
!gcloud config set project {PROJECT_ID}
!gcloud config set compute/region {REGION} --quiet
!gcloud config set compute/zone {ZONE} --quiet

**Enable required services**

In [7]:
!gcloud services enable artifactregistry.googleapis.com
!gcloud services enable run.googleapis.com

**Create the embedEtcd.yaml for Milvus**

In [8]:
embedEtcd = f"""
listen-client-urls: http://0.0.0.0:2379
advertise-client-urls: http://0.0.0.0:2379
quota-backend-bytes: 4294967296
auto-compaction-mode: revision
auto-compaction-retention: '1000'
"""

with open('embedEtcd.yaml', 'w') as file:
    file.write(embedEtcd)

**Create the user.yaml for Milvus**

In [9]:
useryaml = f"""
# Extra config to override default milvus.yaml
"""

with open('user.yaml', 'w') as file:
    file.write(useryaml)

**Create the Dockerfile for Milvus deployment**

In [10]:
Dockerfile = f"""
FROM {IMAGE_NAME}/{IMAGE_TAG}

# Set environment variables
ENV ETCD_USE_EMBED=true
ENV ETCD_DATA_DIR=/var/lib/milvus/etcd
ENV ETCD_CONFIG_PATH=/milvus/configs/embedEtcd.yaml
ENV COMMON_STORAGETYPE=local

# Copy necessary configuration files
#COPY /volumes/milvus /var/lib/milvus
COPY embedEtcd.yaml /milvus/configs/embedEtcd.yaml
COPY user.yaml /milvus/configs/user.yaml

# Expose necessary ports
EXPOSE 19530 9091 2379

# Healthcheck
HEALTHCHECK --interval=30s --start-period=90s --timeout=20s --retries=3 \
  CMD curl -f http://localhost:9091/healthz || exit 1

# Run Milvus standalone
CMD ["milvus", "run", "standalone"]
"""

# Write Dockerfile to a file
with open('Dockerfile', 'w') as file:
    file.write(Dockerfile)

**Create the Artifact Registry repository**

In [ ]:
!gcloud artifacts repositories create {REPO_NAME} \
    --repository-format=docker \
    --location={REGION} \
    --description="Docker repository for Milvus"

**Configure Docker to authenticate with Artifact Registry**

In [ ]:
!gcloud auth configure-docker {REGION}-docker.pkg.dev --quiet

**Build and push the Docker image**

In [ ]:
!gcloud builds submit --tag {REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/{IMAGE_NAME} .

**Deploy and run de container in google cloud run from the image**

In [ ]:
!gcloud run deploy {IMAGE_NAME} \
  --image {REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/{IMAGE_NAME} \
  --platform managed \
  --memory 8Gi \
  --cpu 4 \
  --port {PORT} \
  --region {REGION} \
  --allow-unauthenticated

**Check if eveything is OK**

In [ ]:
import subprocess

# Construct the gcloud command
command = [
    "gcloud", "run", "services", "describe", IMAGE_NAME,
    "--region", REGION,
    "--format", "value(status.url)"
]

# Execute the command
try:
    service_url = subprocess.check_output(command, text=True, stderr=subprocess.STDOUT).strip()
    print(f"🎉 Congratulations 🏆 your container is fully deployed 🎉\n")
    print(f"You can change your params.py accordingly to :")
    print(f"{service_url}:{PORT}")
except subprocess.CalledProcessError as e:
    print(f"An error occurred:\n{e.output}")